# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [1]:
import os
import sys

PROJECT_ROOT = os.getcwd()
sys.path.append(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\HP\Desktop\Assignment1_2026


In [2]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())
import sys
print(sys.executable)

Working directory: c:\Users\HP\Desktop\Assignment1_2026
c:\Users\HP\AppData\Local\Programs\Python\Python312\python.exe


In [3]:
import torch

print("=== Device Check ===")

if torch.cuda.is_available():
    print("Using GPU")
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    device = torch.device("cuda")
else:
    print("Using CPU")
    device = torch.device("cpu")

print("Device:", device)
print("====================")

=== Device Check ===
Using GPU
GPU name: NVIDIA GeForce RTX 3080 Ti Laptop GPU
GPU count: 1
Device: cuda


---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [3]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

Generating train examples…


100%|██████████| 150/150 [00:02<00:00, 61.87it/s]


  30293 questions in total
Generating dev examples…


100%|██████████| 48/48 [00:00<00:00, 53.39it/s]


  10570 questions in total
Generating word embedding…


...

100%|██████████| 30293/30293 [00:03<00:00, 7865.87it/s]


  Built 30169 / 30293 instances
Processing dev examples…


100%|██████████| 10570/10570 [00:01<00:00, 6958.30it/s]


  Built 10465 / 10570 instances
Saving word embedding…
Saving char embedding…
Saving train eval…
Saving dev eval…
Saving word dictionary…
Saving char dictionary…
Saving dev meta…

Preprocessing complete.
  Outputs → _data/


{'train_record_file': '_data\\train.npz',
 'dev_record_file': '_data\\dev.npz',
 'word_emb_file': '_data\\word_emb.json',
 'char_emb_file': '_data\\char_emb.json',
 'train_eval_file': '_data\\train_eval.json',
 'dev_eval_file': '_data\\dev_eval.json',
 'word2idx_file': '_data\\word2idx.json',
 'char2idx_file': '_data\\char2idx.json',
 'dev_meta_file': '_data\\dev_meta.json'}

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [1]:
from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # ── training loop ────────────────────────────────────────────────────
    num_steps  = 60000,
    batch_size = 8,
    seed       = 42,

    # ── vanilla recipe: SGD, no scheduler, NLL loss ───────────────────────
    optimizer_name = "sgd",
    scheduler_name = "step",
    loss_name      = "qa_nll",
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

100%|██████████| 200/200 [00:37<00:00,  5.40it/s]


STEP      200  loss 1819.441761



100%|██████████| 150/150 [00:08<00:00, 17.43it/s]


VALID(train) loss 33.963344  F1 6.926741  EM 0.000000



100%|██████████| 150/150 [00:08<00:00, 17.23it/s]


...

VALID(train) loss 12.130410  F1 7.260100  EM 0.000000



100%|██████████| 150/150 [00:09<00:00, 16.03it/s]


TEST        loss 12.071990  F1 6.532556  EM 0.000000

Learning rate: [0.001]
✓ New best model! F1: 6.5326  EM: 0.0000
Best checkpoint saved to _model/best_model.pt


 44%|████▍     | 88/200 [00:16<00:20,  5.38it/s]


KeyboardInterrupt: 

---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [2]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")

c:\Users\HP\Desktop\Assignment1_2026\EvaluateTools\evaluate.py:118: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=DEVICE)
100%|████

TEST  loss 4.881539  F1 4.627967  EM 0.850454
F1: 4.6280  |  EM: 0.8505  |  Loss: 4.881539
